# HW5: Prompting and Retrieval-Augmented Generation

Run every cell from the top. **Everything already works.**

**Out:** Week 11, Class 1 · **Due:** Week 12, Class 1 · **100 points** · individual work

Work through the notebook and fill in each YOUR TURN cell. Submit this
`.ipynb` with all cells run and their output visible. There is no test to
pass: you are graded on the code working and on your short written answers.

Needs the local model running for the prompting half. If Ollama is not
up you get placeholder replies, and the retrieval half still works.

Today you will:

1. Build an evaluation set FIRST, then measure prompts against it.
2. Build a RAG pipeline and find where retrieval fails.
3. Add the refusal rule that stops it inventing answers.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, logging
from sklearn.metrics.pairwise import cosine_similarity

logging.set_verbosity_error()
enc_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
enc = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2"); enc.eval()

import ollama
def ask(prompt, temperature=0.0):
    try:
        r = ollama.chat(model="qwen2.5:0.5b",
                        messages=[{"role": "user", "content": prompt}],
                        options={"temperature": temperature})
        return r["message"]["content"].strip()
    except Exception:
        return "(no local model running)"

print("setup ready:", ask("Reply with the word ready.")[:30])

## Part 1. Build the evaluation set first (30 points)

The single most common mistake in prompt work is tuning a prompt by eye
and declaring victory. Write the exam before you sit it.

In [ ]:
# ================== YOUR TURN 1 ==================
# Write TESTS: at least 10 (input, expected_label) pairs for a
# three-way routing task of your choosing. Include at least two cases you
# expect to be genuinely hard.
#
# (15 points)
#
# Expected: 10 or more pairs across 3 labels, with at least two you would not be
#           sure about yourself. If every case is obvious your evaluation cannot
#           tell two prompts apart.
# ===============================================
LABELS = ["shipping", "quality", "price"]
TESTS = [
    ("The parcel took three weeks.", "shipping"),
    ("It broke after a month.", "quality"),
    # <-- add at least eight more, including two hard ones
]
print(f"{len(TESTS)} test cases across {len(set(l for _, l in TESTS))} labels")

In [ ]:
# ================== YOUR TURN 2 ==================
# Write score(prompt_template) that runs your test set and returns
# accuracy. Then score one deliberately bad prompt as a baseline.
#
# (15 points)
#
# Expected: a vague prompt should score poorly, near chance (about 0.33 for three
#           labels). If your baseline already scores well, your test set is too
#           easy to measure anything.
# ===============================================
def score(template, show=False):
    """Run template over TESTS and return accuracy."""
    return 0.0          # <-- your code here

BAD_PROMPT = "Categorise this:\n\n{text}"
print(f"baseline accuracy: {score(BAD_PROMPT):.2f}")

## Part 2. Improve the prompt, with evidence (25 points)

Three changes, measured one at a time so you know which one worked.

In [ ]:
# ================== YOUR TURN 3 ==================
# Produce three prompts: one naming the labels, one adding a format
# instruction, one adding two examples. Score all three and chart them.
#
# (25 points)
#
# Expected: naming the labels usually gives the biggest single jump. Examples
#           help less than people expect on a small model. Report what YOU
#           measured, including if it disagrees with that.
# ===============================================
PROMPTS = {
    "baseline": BAD_PROMPT,
    "with labels": None,          # <-- write these
    "with format": None,
    "with examples": None,
}

results = {k: score(v) for k, v in PROMPTS.items() if v is not None}
print(results)
if len(results) > 1:
    plt.figure(figsize=(6, 3))
    plt.bar(list(results), list(results.values()), color="#7C2529")
    plt.ylabel("accuracy"); plt.ylim(0, 1); plt.xticks(rotation=20)
    plt.tight_layout(); plt.show()

# YOUR ANSWER (2 to 3 sentences): which change helped most, and are you
# confident the difference is real given your sample size?
ANSWER_3 = """
"""

## Part 3. RAG (45 points)

Retrieve, then generate. Most failures happen in the first half.

In [ ]:
# GIVEN. A knowledge base and an embedding index.
KNOWLEDGE = [
    "The library is open 8am to midnight on weekdays.",
    "The library closes at 6pm at weekends.",
    "Students may borrow up to 20 books at a time.",
    "Overdue books are fined 25 cents per day, capped at 10 dollars.",
    "Printing costs 8 cents per black and white page.",
    "Group study rooms can be booked online for up to 3 hours.",
    "The archive on floor 4 requires an appointment.",
    "Laptops may be borrowed for 4 hours from the front desk.",
]

@torch.no_grad()
def embed(texts):
    ids = enc_tok(texts, return_tensors="pt", padding=True, truncation=True)
    return enc(**ids).last_hidden_state.mean(dim=1).numpy()

index = embed(KNOWLEDGE)
print(f"{len(KNOWLEDGE)} facts indexed, {index.shape[1]} dimensions each")

In [ ]:
# ================== YOUR TURN 4 ==================
# Write retrieve(question, k) returning the k closest facts WITH their
# similarity scores, then rag(question) that answers using only those.
#
# (20 points)
#
# Expected: 'How much is printing?' retrieves the printing fact at above 0.6 and
#           answers correctly. Print the scores: you need them for the next part.
# ===============================================
def retrieve(question, k=2):
    """Return [(fact, score), ...] for the k closest facts."""
    return []          # <-- your code here

def rag(question, k=2):
    passages = retrieve(question, k)
    if not passages:
        return "(retrieve is not implemented yet)", []
    context = "\n".join(f"- {t}" for t, s in passages)
    prompt = (f"Answer using ONLY these facts:\n{context}\n\n"
              f"Question: {question}\nAnswer in one short sentence.")
    return ask(prompt), passages

answer, used = rag("How much is printing?")
for t, s in used:
    print(f"   {s:.3f}  {t}")
print("answer:", answer)

In [ ]:
# ================== YOUR TURN 5 ==================
# Find a question the knowledge base CANNOT answer. Show what gets
# retrieved and what the model says.
#
# (10 points)
#
# Expected: retrieval returns its nearest facts regardless, at a much lower
#           score, and the model answers from them anyway. Record the score: it is
#           the signal you need for the fix.
# ===============================================
UNANSWERABLE = "What is the wifi password?"          # <-- change if you like

answer, used = rag(UNANSWERABLE)
for t, s in used:
    print(f"   {s:.3f}  {t}")
print("answer:", answer)

In [ ]:
# ================== YOUR TURN 6 ==================
# Add a similarity threshold so the system refuses instead of
# inventing. Choose the value from YOUR measurements and justify it.
#
# (15 points)
#
# Expected: a threshold between your answerable scores and your unanswerable one.
#           Marks are for choosing it from measured numbers rather than picking a
#           round figure, and for saying what it costs when you set it too high.
# ===============================================
THRESHOLD = 0.0          # <-- choose from your measurements

QUESTIONS = ["How much is printing?", "How many books can I borrow?",
             "When does it close at the weekend?", UNANSWERABLE]

for q in QUESTIONS:
    passages = retrieve(q, k=2)
    best = passages[0][1] if passages else 0.0
    verdict = "REFUSED " if best < THRESHOLD else "answered"
    print(f"   {verdict} ({best:.3f})  {q}")

# YOUR ANSWER (3 to 4 sentences): why this threshold, and what does a too-high
# value cost you?
ANSWER_6 = """
"""

## Answers

Try each task before reading.

In [ ]:
# Marking
#   Q1  15   10+ cases, 3 labels, at least two genuinely hard
#   Q2  15   scorer works; a weak baseline scores near chance
#   Q3  25   three prompts measured separately and charted
#   Q4  20   retrieval and generation both working
#   Q5  10   an unanswerable question demonstrated with its score
#   Q6  15   threshold chosen from measurements and defended
#
# Where students lose marks:
#   - writing the test set AFTER tuning the prompt, which measures nothing
#   - reporting a 0.1 accuracy difference on 10 items as if it were real
#   - picking THRESHOLD = 0.5 because it is a round number rather than because
#     it sits between the scores they measured